# persocial · Colab scorer

Scores comments (sentiment + emotion) on Colab's GPU. Two ways to use it:

**A · CSV (works with the local database, no cloud needed)**
1. Web app → Scoring → **Export next post as CSV**.
2. Here: **Runtime → Change runtime type → T4 GPU** → run the first cells, then the
   **CSV mode** cell → choose the CSV → the scored file downloads.
3. Web app → Upload → drop the scored CSV. Scores attach and the post's lease is released.

**B · Direct (cloud database)** — set the `COLAB_API_URL` and `COLAB_TOKEN` secrets
(🔑 on the left), run all, use the form at the bottom. Scores are saved to the cloud
every *Save every* comments.

In [ ]:
%pip install -q transformers pandas ipywidgets requests
import json, time, requests, torch
from google.colab import userdata, output
output.enable_custom_widget_manager()
try:                       # only needed for mode B (direct to the cloud database)
    API_URL, TOKEN = userdata.get("COLAB_API_URL"), userdata.get("COLAB_TOKEN")
except Exception:
    API_URL = TOKEN = None
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — CPU will be slow")

In [ ]:
%%writefile scoring_core.py
"""Sentiment + emotion scoring. SELF-CONTAINED on purpose: scripts/build_colab_notebook.py
embeds this file verbatim into colab/score_csv.ipynb, so it may import only the stdlib,
torch, transformers and pandas — never anything from `persocial`.

Matches the historical scorer (sentiment_fast_hybrid_local_v2.py) so new scores are
comparable with earlier runs: same models, same @user/http
preprocessing, all sentiment labels ranked, emotions kept only above 0.45.
"""

from __future__ import annotations

import json
import re
import time
from datetime import UTC, datetime

MODEL_VERSION = "persocial-v1"
SENTIMENT_MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual"
EMOTION_MODEL = "AnasAlokla/multilingual_go_emotions"
EMOTION_THRESHOLD = 0.45
MAX_LENGTH = 512
DEFAULT_BATCH = {"cuda": 128, "mps": 48, "cpu": 64}   # MPS stalls on large batches

SCORE_COLUMNS = ["sentiment_fast", "emotions_fast", "updated_at", "model_version"]


def preprocess(text: str) -> str:
    t = re.sub(r"(?<!\S)@\S+", "@user", text or "")
    return re.sub(r"http\S*", "http", t)


def resolve_device(prefer: str = "auto"):
    import torch
    if prefer in ("cuda", "auto") and torch.cuda.is_available():
        return "cuda", 0, torch.float16, torch.cuda.get_device_name(0)
    if prefer in ("mps", "auto") and torch.backends.mps.is_available():
        # fp32 on Metal: fp16 kernels fall back silently and end up slower
        return "mps", "mps", torch.float32, "Apple Metal"
    return "cpu", -1, torch.float32, "CPU"


class Scorer:
    def __init__(self, device: str = "auto", batch_size: int = 0):
        from transformers import AutoConfig, pipeline
        self.device, pipe_device, dtype, self.device_name = resolve_device(device)
        self.batch_size = batch_size or DEFAULT_BATCH[self.device]
        common = {"device": pipe_device, "torch_dtype": dtype, "truncation": True,
                  "max_length": MAX_LENGTH}
        self.sent = pipeline("text-classification", model=SENTIMENT_MODEL, top_k=None, **common)
        n_emo = AutoConfig.from_pretrained(EMOTION_MODEL).num_labels
        self.emo = pipeline("text-classification", model=EMOTION_MODEL, top_k=n_emo, **common)

    @property
    def runner(self) -> str:
        return f"{self.device}:{self.device_name}"

    def score(self, texts: list[str], on_progress=None) -> list[tuple[list[dict], list[dict]]]:
        """[(sentiment ranked, emotions above threshold)] per text, same order."""
        clean = [preprocess(t) or " " for t in texts]
        out: list[tuple[list[dict], list[dict]]] = []
        for start in range(0, len(clean), self.batch_size):
            batch = clean[start:start + self.batch_size]
            s_res = self.sent(batch, batch_size=len(batch))
            e_res = self.emo(batch, batch_size=len(batch))
            for s_row, e_row in zip(s_res, e_res):
                sent = sorted(({"label": r["label"], "score": round(float(r["score"]), 4)}
                               for r in s_row), key=lambda x: x["score"], reverse=True)
                emo = sorted(({"label": r["label"], "score": round(float(r["score"]), 4)}
                              for r in e_row if r["score"] >= EMOTION_THRESHOLD),
                             key=lambda x: x["score"], reverse=True)
                out.append((sent, emo))
            if on_progress:
                on_progress(min(start + self.batch_size, len(clean)), len(clean))
        return out


def score_csv(in_path: str, out_path: str, device: str = "auto", batch_size: int = 0) -> dict:
    """Colab / CLI entry: read an exported CSV, fill the score columns for rows that lack
    them, write the result. Every other column (incl. claim_id) passes through untouched,
    so the output imports straight back through the normal upload."""
    import pandas as pd
    df = pd.read_csv(in_path, dtype=str, keep_default_na=False)
    for col in SCORE_COLUMNS:
        if col not in df.columns:
            df[col] = ""
    todo = df.index[df["sentiment_fast"].str.strip() == ""].tolist()
    t0 = time.time()
    if todo:
        scorer = Scorer(device, batch_size)
        print(f"scoring {len(todo)} comments on {scorer.runner}")

        def progress(done, total):
            print(f"  {done}/{total}  {done / max(time.time() - t0, 1e-6):.0f}/s", flush=True)

        results = scorer.score(df.loc[todo, "text"].tolist(), on_progress=progress)
        now = datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S UTC")
        for idx, (sent, emo) in zip(todo, results):
            df.at[idx, "sentiment_fast"] = json.dumps(sent, ensure_ascii=False)
            df.at[idx, "emotions_fast"] = json.dumps(emo, ensure_ascii=False)
            df.at[idx, "updated_at"] = now
            df.at[idx, "model_version"] = MODEL_VERSION
    df.to_csv(out_path, index=False, encoding="utf-8")
    return {"rows": len(df), "scored": len(todo), "seconds": round(time.time() - t0, 1)}


In [ ]:
# ── Mode A: score a CSV exported from the web app (Scoring → Export next post as CSV) ──
from google.colab import files
from scoring_core import score_csv
for name, data in files.upload().items():
    open(name, "wb").write(data)
    out = name.rsplit(".", 1)[0] + "-scored.csv"
    print(name, "→", out, score_csv(name, out))
    files.download(out)       # then: web app → Upload → drop this file

In [ ]:
from scoring_core import Scorer

def api(action, **kw):
    r = requests.post(API_URL, headers={"x-colab-token": TOKEN},
                      json={"action": action, **kw}, timeout=300)
    if r.status_code != 200:
        raise RuntimeError(f"{action}: {r.status_code} {r.text[:300]}")
    return r.json()

_scorer = None
def scorer():
    global _scorer
    if _scorer is None:
        _scorer = Scorer("auto")
    return _scorer

def run(post_codes=None, save_every=5000, log=print, bar=None):
    """Lease one post at a time; score its unscored comments in save_every chunks; each
    submit is a save point that keeps the lease alive. Stops when nothing is left."""
    s = scorer()
    runner = f"colab:{s.device_name}"
    total = 0
    while True:
        leases = api("claim", post_codes=post_codes, n=1, runner=runner)
        if not leases:
            log(f"✅ done — nothing left to score. {total} comments scored this session.")
            return total
        code, todo = leases[0]["post_code"], leases[0]["n_unscored"]
        log(f"▶ post {code}: {todo} comments")
        done = 0
        try:
            while True:
                chunk = api("comments", post_code=code, limit=save_every)
                if not chunk:
                    break
                t0 = time.time()
                res = s.score([c["text"] for c in chunk])
                rows = [{"ig_id": c["ig_id"], "sentiment": a, "emotions": b}
                        for c, (a, b) in zip(chunk, res)]
                out = api("submit", post_code=code, rows=rows, runner=runner)
                done += out["written"]; total += out["written"]
                rate = len(chunk) / max(time.time() - t0, 1e-6)
                log(f"   saved {done}/{todo} · {rate:.0f}/s · {out['remaining']} left")
                if bar is not None:
                    bar.value = done / max(todo, 1)
                if out["remaining"] == 0:
                    break
        except Exception as e:
            api("release", post_code=code)
            log(f"✖ {code}: {e} — lease released, run again to retry")
            raise
        if post_codes:
            post_codes = [p for p in post_codes if p != code] or None
            if post_codes is None:
                log(f"✅ done — selected posts scored. {total} comments this session.")
                return total

In [ ]:
# ── Mode B: direct to the cloud database (needs COLAB_API_URL / COLAB_TOKEN secrets) ──
if not API_URL:
    raise SystemExit("Mode B needs the COLAB_API_URL and COLAB_TOKEN secrets — use mode A above instead.")
import ipywidgets as W
from IPython.display import display

pending = api("pending")
n_left = sum(p["n_unscored"] for p in pending)
scope = W.Dropdown(options=[(f"All unanalysed comments ({n_left:,} in {len(pending)} posts)", "all"),
                            ("Pick posts", "pick")], description="Scope", layout=W.Layout(width="520px"))
search = W.Text(placeholder="search post id or account", description="Search", layout=W.Layout(width="520px"))
def _opts(q=""):
    q = q.lower()
    return [(f"{p['code']} · @{p['username'] or '?'} · {p['n_unscored']:,} left"
             + (" · claimed" if p["claimed"] else ""), p["code"])
            for p in pending if not q or q in p["code"].lower() or q in (p["username"] or "").lower()]
posts = W.SelectMultiple(options=_opts(), rows=10, description="Posts", layout=W.Layout(width="520px"))
save_every = W.Dropdown(options=[("1,000", 1000), ("5,000", 5000), ("10,000", 10000)], value=5000,
                        description="Save every")
start = W.Button(description="Start", button_style="success", icon="play")
bar = W.FloatProgress(min=0, max=1, description="Post")
log = W.Output(layout=W.Layout(border="1px solid #ddd", max_height="320px", overflow="auto"))
pick_box = W.VBox([search, posts])
pick_box.layout.display = "none"
scope.observe(lambda c: setattr(pick_box.layout, "display", "" if c["new"] == "pick" else "none"), "value")
search.observe(lambda c: setattr(posts, "options", _opts(c["new"])), "value")

def _go(_):
    start.disabled = True
    codes = list(posts.value) if scope.value == "pick" else None
    if scope.value == "pick" and not codes:
        with log: print("pick at least one post"); start.disabled = False; return
    with log:
        try:
            run(codes, save_every.value, log=print, bar=bar)
        finally:
            start.disabled = False
start.on_click(_go)
display(W.VBox([scope, pick_box, save_every, start, bar, log]))